# Feature store
Puede crear FS de dos formas
- Adicione a una tabla PK. 
- Usar la función `FeatureEngineeringClient`

Vamos a crear tablas de caracteristicas:
* **User features**: features for a given user in a given point in time (location, previous purchases if any, tenure days etc)
* **Destination features**: data on the travel destination for a given point in time (interest tracked by the number of clicks & impression)

### Soporte para punto en el tiempo para tablas de caracteristicas
Databricks FS soporta casos de usos que requiere exactitud en un punto en el tiempo. Los datos a menudo tienen dependencia del tiempo. Para este ejemplo adicionamos rolling-window features. Al construir el modelo, debemos considerar solo valores de características hasta el tiempo de los valores observados. De lo contrario puede tomar valores después de la marca de tiempo de los valores a entrenar. Esto es llamado fuga de datos “data leakage” y puede afectar negativamente el desempeño del modelo. Tablas de caracteristicas serie de tiempo incluye una columna clave (key) para identificar el tiempo.

**Prompt para crear FS con AI:**

Contexto: 
Dataframe: dataset_df, que toma datos de vm_churn_feature

Objetivo:
Crear tabla Fecture store y poblar datos con el dataframe: dataset_df

Reglas:
- Uso de FeatureStoreClient
- Propiedad primary_keys = user_id
- Propiedad schema = dataset_df.spark.schema()
- Propiedad description = Esta FS es derivada de la tabla churn_user_bronze > vm_churn_feature. No tiene agregaciones.
 
Modo de trabajo:
- Validar si existe la tabla antes de crearla
- Guardar la tabla en el mismo catalogo y esquema de la tabla base del dataframe: dataset_df.
- Realizar la creacion de la tabla y copiado de datos en dos instrucciones diferentes.

Formato de respuesta:
- Nombre de la tabla fecture store: churn_user_features

In [0]:
%pip install databricks-feature-engineering
dbutils.library.restartPython()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.2/866.2 kB 27.4 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.4
    Not uninstalling protobuf at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-682c1ff2-c5c2-411e-8d04-604db6e67309
    Can't uninstall 'protobuf'. No files were found to uninstall.
  Attempting uninstall: databricks-sdk
    Found existing installation: databricks-sdk 0.67.0
    Not uninstalling databricks-sdk at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-682c1ff2-c5c2-411e-8d04-604db6e67309
    Can't uninstall 'databricks-sdk'. No files were found to uninstall.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
catalog = "medallion_dev"
schema = "gold"

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"""USE `{catalog}`.`{schema}`""")

DataFrame[]

In [0]:
dataset_df = spark.table(f"{catalog}.{schema}.vm_churn_feature")
display(dataset_df.limit(5))

# Borrar columnas no relevantes para el modelo
dataset_df = dataset_df.drop('address', 'email', 'firstname', 'lastname', 'creation_date', 'last_activity_date', 'last_event')
# borrar valores ausentes
dataset_df = dataset_df.dropna()
display(dataset_df.limit(5))

address,age_group,canal,churn,country,creation_date,email,firstname,gender,user_id,last_activity_date,lastname,ingest_datetime_silver,platform,event_count,session_count,last_event,order_count,total_amount,total_item,last_transaction,days_since_creation,days_since_last_activity,days_last_event
"64827 Johnson Streets Apt. 420 Garzaville, LA 10417",4,PHONE,1,SPAIN,2023-01-11T00:00:00.000Z,welchthomas@henry.com,Nicole,0,f617a497-9026-445d-b077-f06f2e3aa82c,2023-06-06T14:00:33.000Z,Andrade,2026-04-10T21:26:14.553Z,ios,4,3,2023-06-06T03:55:52.000Z,4,223,9,2023-06-05T06:23:07.000Z,1185,1039,1039
Unit 6945 Box 6919 DPO AP 59513,1,MOBILE,0,FR,2023-02-25T00:00:00.000Z,shane38@myers.biz,Ashley,1,ee8a18a8-eebf-40cd-a46d-5a2474089c64,2023-06-09T04:54:55.000Z,Armstrong,2026-04-10T21:26:14.553Z,android,6,5,2023-06-06T19:19:22.000Z,6,247,10,2023-06-09T06:33:02.000Z,1140,1036,1039
"9587 Caldwell Meadow East Christine, MA 11986",5,PHONE,1,USA,2022-07-03T00:00:00.000Z,kingcody@moody.com,Jessica,1,5f52e274-e0cd-4665-9c15-05590385692c,2023-06-02T01:42:11.000Z,Daniel,2026-04-10T21:26:14.553Z,other,1,0,2023-06-04T10:39:09.000Z,1,96,3,2023-06-08T02:08:17.000Z,1377,1043,1041
Unit 3634 Box 5928 DPO AE 54813,7,MOBILE,1,USA,2022-01-02T00:00:00.000Z,lynnford@goodwin-graham.com,Cheryl,1,35bd5a18-66a5-482c-b618-e353128e67fe,2023-06-06T21:33:07.000Z,Salazar,2026-04-10T21:26:14.553Z,other,4,3,2023-06-09T00:33:37.000Z,4,191,7,2023-06-09T18:28:14.000Z,1559,1039,1036
"48351 Colleen Drives Suite 373 Danielborough, LA 16110",5,WEBAPP,0,SPAIN,2022-03-13T00:00:00.000Z,andersonmichael@cook.com,Jeffrey,1,509b3da6-c25a-4b9a-b375-5ade442b95c2,2023-06-04T23:26:33.000Z,Walker,2026-04-10T21:26:14.553Z,ios,2,1,2023-06-07T17:39:33.000Z,2,103,5,2023-06-08T21:50:54.000Z,1489,1041,1038


age_group,canal,churn,country,gender,user_id,ingest_datetime_silver,platform,event_count,session_count,order_count,total_amount,total_item,last_transaction,days_since_creation,days_since_last_activity,days_last_event
4,PHONE,1,SPAIN,0,f617a497-9026-445d-b077-f06f2e3aa82c,2026-04-10T21:26:14.553Z,ios,4,3,4,223,9,2023-06-05T06:23:07.000Z,1185,1039,1039
1,MOBILE,0,FR,1,ee8a18a8-eebf-40cd-a46d-5a2474089c64,2026-04-10T21:26:14.553Z,android,6,5,6,247,10,2023-06-09T06:33:02.000Z,1140,1036,1039
5,PHONE,1,USA,1,5f52e274-e0cd-4665-9c15-05590385692c,2026-04-10T21:26:14.553Z,other,1,0,1,96,3,2023-06-08T02:08:17.000Z,1377,1043,1041
7,MOBILE,1,USA,1,35bd5a18-66a5-482c-b618-e353128e67fe,2026-04-10T21:26:14.553Z,other,4,3,4,191,7,2023-06-09T18:28:14.000Z,1559,1039,1036
5,WEBAPP,0,SPAIN,1,509b3da6-c25a-4b9a-b375-5ade442b95c2,2026-04-10T21:26:14.553Z,ios,2,1,2,103,5,2023-06-08T21:50:54.000Z,1489,1041,1038


### Crear Feature Table
Usamos FeatureStore client para creear tablas FS. Note el parametro `timestamp_keys='ts'` usado como columna de tiempo. Databricks Feature Store usa esta informacion para filtrar features y prevenir potenciales fugas de datos (leakage).

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient

fe = FeatureEngineeringClient()
feature_table_name = f"{catalog}.{schema}.churn_user_features"
try:
    existing_table = fe.get_table(name=feature_table_name)
    print(f"La tabla {feature_table_name} ya existe.")
except Exception as e:
    print(f"La tabla no existe, procediendo a crearla...")
    
    # Crear la tabla Feature Store
    fe.create_table(
        name=feature_table_name,
        primary_keys=["user_id"],
        schema=dataset_df.schema,
        description="Esta FS es derivada de la tabla churn_user_bronze > vm_churn_feature. No tiene agregaciones."
    )
    print(f"Tabla {feature_table_name} creada exitosamente.")

# Poblar la tabla con datos del dataframe
fe.write_table(name=feature_table_name, df=dataset_df, mode="merge")

print(f"Datos escritos en la tabla {feature_table_name}")

La tabla no existe, procediendo a crearla...


2026/04/27 04:12:26 INFO databricks.ml_features._compute_client._compute_client: Setting columns ['user_id'] of table 'medallion_dev.gold.churn_user_features' to NOT NULL.
2026/04/27 04:12:28 INFO databricks.ml_features._compute_client._compute_client: Setting Primary Keys constraint ['user_id'] on table 'medallion_dev.gold.churn_user_features'.
2026/04/27 04:12:29 INFO databricks.ml_features._compute_client._compute_client: Created feature table 'medallion_dev.gold.churn_user_features'.


Tabla medallion_dev.gold.churn_user_features creada exitosamente.
Datos escritos en la tabla medallion_dev.gold.churn_user_features


In [0]:
spark.sql(f"""ALTER TABLE {feature_table_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)""")
spark.sql(f"""ALTER TABLE {feature_table_name} ALTER COLUMN user_id SET NOT NULL""")

DataFrame[]

In [0]:
fe_table = fe.get_table(name=feature_table_name)
print(f"Feature Table: {fe_table.name}. Descripcion: {fe_table.description}")
print("Caracteristicas: ", fe_table.features)

Feature Table: medallion_dev.gold.churn_user_features. Descripcion: Esta FS es derivada de la tabla churn_user_bronze > vm_churn_feature. No tiene agregaciones.
La tabla contiene estas caracteristicas:  ['age_group', 'canal', 'churn', 'country', 'gender', 'user_id', 'ingest_datetime_silver', 'platform', 'event_count', 'session_count', 'order_count', 'total_amount', 'total_item', 'last_transaction', 'days_since_creation', 'days_since_last_activity', 'days_last_event']
